# Global Analysis

In [1]:
def default_params(): 
    return {
        'current_model': 'M1', 
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'name': 'blastwind/github-code-haskell-function',
            'content_column': 'full_solution', 
            'number_samples': 55,
        },
        'probs_bootstrapping_per_node':500,
        'default_max_position_embeddings' : 1024,
        'output_path' : '/workspaces/csci-635_functional_interpretability/data/global_analysis',
        'logits_path': '../data/raw_logits',
        'aggregation_path': '../data/aggregated_nodes',
        'preprocessed_dataset_dir' : '../datax/functional_interpretability/dataset_preprocessing',
        'cache_dir': '../datax/hugging_face_cache',
        'log_file': '../datax/functional_interpretability/logit_extraction.log', 
        'callbacks_dir' : '../datax/functional_interpretability/callbacks',
        'causal_models': {
            'M1': 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2': 'codellama/CodeLlama-13b-hf', #https://huggingface.co/codellama/CodeLlama-13b-hf
            'M3': 'codellama/CodeLlama-34b-hf', # https://huggingface.co/codellama/CodeLlama-34b-hf
            'M4': 'codellama/CodeLlama-70b-hf', #https://huggingface.co/codellama/CodeLlama-70b-hf, 
            'M5': 'mistralai/Mistral-7B-v0.1', #https://huggingface.co/mistralai/Mistral-7B-v0.,
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
from datasets import load_dataset 
from statistics import mean, median
import json
import torch
import gc

In [3]:
from functional_interpretability_lib.loader import download_grammars
from functional_interpretability_lib.parser import create_parser

In [4]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, pipeline, Conversation, BitsAndBytesConfig, CodeLlamaTokenizer, LlamaForCausalLM, PreTrainedTokenizerFast, CodeLlamaTokenizerFast

2024-04-05 20:17:24.922211: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-04-05 20:17:24.922284: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-04-05 20:17:24.923585: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-04-05 20:17:24.930541: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
import logging
#logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
logging.basicConfig(
    filename=params['log_file'],
    filemode='a',
    format='%(asctime)s : %(levelname)s : %(message)s', 
    level=logging.INFO
    )

#### Model Loading

In [6]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = CodeLlamaTokenizerFast.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     match params['quantization']:
               case 'int4':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
               case 'int8':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
               case 'float32':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
               case 'float16':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
               case _: 
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [7]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'LlamaTokenizer'. 
The class this function is called from is 'CodeLlamaTokenizerFast'.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

#### Parsers

In [8]:
download_grammars(['haskell'])
parser, node_types, language = create_parser('haskell')

/usr/local/lib/python3.11/dist-packages/functional_interpretability_lib/grammars


#### Load Aggregates

In [9]:
df_actual_ntp = pd.read_csv(params['aggregation_path'] + '/' + params['current_model'] + '_q_' + params['quantization'] + '/' + 'aggregated_nodes.csv', index_col=0)

FileNotFoundError: [Errno 2] No such file or directory: '../data/aggregated_nodes/M5_q_float16/aggregated_nodes.csv'

In [10]:
df_actual_ntp.head(3)

,problem_type,problem,full_solution,entry_point,function,signature,context,unit_test_template,input_ids,max_prob,min_prob,actual_prob,loss,binded_tree
0,function,The function `calculateDiscount` takes a singl...,calculateDiscount :: Int -> Int\ncalculateDisc...,calculateDiscount,calculateDiscount amount\n | amount < 100 = a...,calculateDiscount :: Int -> Int,NaN,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[13911, 3278, 2114, 6210, 4666, 3193, 4666, 13...","[('#', 0.2821720838546753), ('_', 0.2682041525...","[('–,', 1.963395279691582e-10), ('même', 1.596...","[('calculate', 1.8452654160228121e-07), ('Dis'...",0.916391,"{'type': 'haskell', 'children': [{'type': 'sig..."
1,function,The function `calculateThemeChange` takes two ...,calculateThemeChange :: String -> String -> St...,calculateThemeChange,calculateThemeChange current desired\n | curr...,calculateThemeChange :: String -> String -> St...,NaN,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[13911, 10438, 5500, 6210, 1677, 3193, 1677, 3...","[('#', 0.2821975648403168), ('_', 0.2699084877...","[('–,', 1.963572499041888e-10), ('même', 1.593...","[('calculate', 1.847235040486339e-07), ('Theme...",1.698175,"{'type': 'haskell', 'children': [{'type': 'sig..."
2,function,The function `updatePath` takes two strings as...,import Data.List (isInfixOf)\n\nupdatePath :: ...,updatePath,updatePath existingPath newDir\n | newDir `is...,updatePath :: String -> String -> String,import Data.List (isInfixOf)\n\n,import Data.List (isInfixOf)\nimport Test.Hspe...,"[726, 5284, 28723, 1245, 325, 278, 657, 7192, ...","[('#', 0.28298500180244446), ('{', 0.311360031...","[('–,', 1.9537282902604147e-10), ('/******/', ...","[('import', 0.023048054426908493), ('Data', 0....",1.366822,"{'type': 'haskell', 'children': [{'type': 'imp..."


#### Parent and Children types extraction

In [11]:
def traverse_binded_tree_and_fill_hierarchy(binded_tree: dict):
    parent_nodes = []
    leaf_nodes = []
    def traverse_binded_tree(binded_tree: dict, parent_nodes:list, leaf_nodes: list):
        if binded_tree['children']:
            parent_nodes.append(binded_tree['type'])
            for kid in binded_tree['children']:
                traverse_binded_tree(kid, parent_nodes, leaf_nodes)
        if not binded_tree['children']:
            leaf_nodes.append(binded_tree['type'])
    traverse_binded_tree(binded_tree, parent_nodes, leaf_nodes)
    return set(parent_nodes), set(leaf_nodes)

In [12]:
def get_parent_and_leaf_nodes(df_actual_ntp):
    binded_trees = df_actual_ntp['binded_tree'].values
    parent_nodes = set()
    leaf_nodes = set()
    for binded_tree in binded_trees:
        a, b = traverse_binded_tree_and_fill_hierarchy(eval(binded_tree))
        parent_nodes.update(a)
        leaf_nodes.update(b)
    return parent_nodes, leaf_nodes

In [13]:
parent_node_types, child_node_types = get_parent_and_leaf_nodes(df_actual_ntp)

#### Local Analysis (Snippets)

In [14]:
def traverse_tree_and_collect_stds(node: dict, node_types_list: list, std_field: str, grammar_node_types: list):
    if node[std_field] is not None:
        node_types_list[grammar_node_types.index(node['type'])] = node_types_list[grammar_node_types.index(node['type'])] + [node[std_field]]
    for child in node['children']:
        traverse_tree_and_collect_stds(child, node_types_list, std_field, grammar_node_types)

In [15]:
def add_statistic_column(std_field, dataframe, grammar_node_types):
    concept_probs = []
    for tree in dataframe['binded_tree']:
        node_types_list = [[] for type in grammar_node_types]
        traverse_tree_and_collect_stds(eval(tree), node_types_list, std_field, grammar_node_types)
        snippet_type_list = []
        for type_index, node_values in enumerate(node_types_list):
            if len(node_values)>0: 
                snippet_type_list.append((grammar_node_types[type_index], node_values))
        concept_probs.append(snippet_type_list)
    dataframe['concept_'+std_field] =  concept_probs

In [16]:
add_statistic_column('median_prob', df_actual_ntp, node_types)
add_statistic_column('min_prob', df_actual_ntp, node_types)
add_statistic_column('max_prob', df_actual_ntp, node_types)
df_actual_ntp.head()

,problem_type,problem,full_solution,entry_point,function,signature,context,unit_test_template,input_ids,max_prob,min_prob,actual_prob,loss,binded_tree,concept_median_prob,concept_min_prob,concept_max_prob
0,function,The function `calculateDiscount` takes a singl...,calculateDiscount :: Int -> Int\ncalculateDisc...,calculateDiscount,calculateDiscount amount\n | amount < 100 = a...,calculateDiscount :: Int -> Int,NaN,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[13911, 3278, 2114, 6210, 4666, 3193, 4666, 13...","[('#', 0.2821720838546753), ('_', 0.2682041525...","[('–,', 1.963395279691582e-10), ('même', 1.596...","[('calculate', 1.8452654160228121e-07), ('Dis'...",0.916391,"{'type': 'haskell', 'children': [{'type': 'sig...","[(patterns, [0.998976469039917]), (guard_equat...","[(patterns, [0.998976469039917]), (guard_equat...","[(patterns, [0.998976469039917]), (guard_equat..."
1,function,The function `calculateThemeChange` takes two ...,calculateThemeChange :: String -> String -> St...,calculateThemeChange,calculateThemeChange current desired\n | curr...,calculateThemeChange :: String -> String -> St...,NaN,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[13911, 10438, 5500, 6210, 1677, 3193, 1677, 3...","[('#', 0.2821975648403168), ('_', 0.2699084877...","[('–,', 1.963572499041888e-10), ('même', 1.593...","[('calculate', 1.847235040486339e-07), ('Theme...",1.698175,"{'type': 'haskell', 'children': [{'type': 'sig...","[(patterns, [0.5166219212114811]), (guard_equa...","[(patterns, [0.03399205952882767]), (guard_equ...","[(patterns, [0.9992517828941345]), (guard_equa..."
2,function,The function `updatePath` takes two strings as...,import Data.List (isInfixOf)\n\nupdatePath :: ...,updatePath,updatePath existingPath newDir\n | newDir `is...,updatePath :: String -> String -> String,import Data.List (isInfixOf)\n\n,import Data.List (isInfixOf)\nimport Test.Hspe...,"[726, 5284, 28723, 1245, 325, 278, 657, 7192, ...","[('#', 0.28298500180244446), ('{', 0.311360031...","[('–,', 1.9537282902604147e-10), ('/******/', ...","[('import', 0.023048054426908493), ('Data', 0....",1.366822,"{'type': 'haskell', 'children': [{'type': 'imp...","[(patterns, [0.4367777110892348, 0.51138724666...","[(patterns, [0.0017303533386439085, 0.02377710...","[(patterns, [0.9993153810501099, 0.99899739027..."
3,function,Design a function named `calculateDiscount` th...,calculateDiscount :: Float -> Int -> Bool -> F...,calculateDiscount,calculateDiscount price discountPercentage onC...,calculateDiscount :: Float -> Int -> Bool -> F...,NaN,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[13911, 3278, 2114, 6210, 27914, 3193, 4666, 3...","[('#', 0.28301000595092773), ('_', 0.269829601...","[('–,', 1.938695731729112e-10), ('même', 1.605...","[('calculate', 1.8184967132128804e-07), ('Dis'...",0.848477,"{'type': 'haskell', 'children': [{'type': 'sig...","[(patterns, [0.3224333902554853, 0.99990379810...","[(patterns, [0.011824269512934344, 0.999903798...","[(patterns, [0.7765514207671264, 0.99990379810..."
4,function,The function `filterAndCount` takes a list of ...,import Data.List (filter)\n\nfilterAndCount ::...,filterAndCount,filterAndCount strs char threshold = length $ ...,filterAndCount :: [String] -> Char -> Int -> Int,import Data.List (filter)\n\n,import Data.List (filter)\nimport Test.Hspec\n...,"[726, 5284, 28723, 1245, 325, 4650, 28731, 13,...","[('#', 0.28298500180244446), ('{', 0.311360031...","[('–,', 1.9537282902604147e-10), ('/******/', ...","[('import', 0.023048054426908493), ('Data', 0....",0.962341,"{'type': 'haskell', 'children': [{'type': 'imp...","[(patterns, [0.4252041559666395, 0.61459541320...","[(patterns, [0.04634538292884827, 0.3112695813...","[(patterns, [0.999921441078186, 0.999504327774..."


### Global Analysis (AST Elements)

In [17]:
global_concept_dataframe = pd.DataFrame([], columns=['ast_element', 'node_type' ,'concept_median_prob', 'concept_min_prob','concept_max_prob'])
for concept_idx in range(0,len(node_types)):
    global_concept_dataframe.loc[len(global_concept_dataframe.index)] = [node_types[concept_idx],
                                                                             'parent' if node_types[concept_idx] in parent_node_types else 'leaf',
                                                                             [], 
                                                                             [], 
                                                                             []]
for tree in df_actual_ntp['binded_tree']:
    concept_median_prob_list = [[] for type in node_types]
    concept_min_prob_list = [[] for type in node_types]
    concept_max_prob_list = [[] for type in node_types]
    traverse_tree_and_collect_stds(eval(tree), concept_median_prob_list, 'median_prob', node_types)
    traverse_tree_and_collect_stds(eval(tree), concept_min_prob_list, 'min_prob', node_types)
    traverse_tree_and_collect_stds(eval(tree), concept_max_prob_list, 'max_prob', node_types)
    for concept_idx in range(0,len(node_types)):
        global_concept_dataframe.at[concept_idx, 'concept_median_prob'] = global_concept_dataframe['concept_median_prob'][concept_idx] + concept_median_prob_list[concept_idx]
        global_concept_dataframe.at[concept_idx, 'concept_min_prob'] = global_concept_dataframe['concept_min_prob'][concept_idx] + concept_min_prob_list[concept_idx]
        global_concept_dataframe.at[concept_idx, 'concept_max_prob'] = global_concept_dataframe['concept_max_prob'][concept_idx] + concept_max_prob_list[concept_idx]

global_concept_dataframe = global_concept_dataframe.drop(global_concept_dataframe[global_concept_dataframe['concept_median_prob'].map(len) == 0].index)

In [18]:
global_concept_dataframe

,ast_element,node_type,concept_median_prob,concept_min_prob,concept_max_prob
0,patterns,parent,"[0.998976469039917, 0.5166219212114811, 0.4367...","[0.998976469039917, 0.03399205952882767, 0.001...","[0.998976469039917, 0.9992517828941345, 0.9993..."
3,guard_equation,parent,"[0.696838410364257, 0.7294911894598044, 0.9376...","[0.40085938572883606, 0.45910379581619054, 0.8...","[0.9421604143248664, 0.9223121285438538, 0.997..."
10,module,leaf,"[0.023048054426908493, 0.5911130309104919, 0.0...","[0.023048054426908493, 0.5911130309104919, 0.0...","[0.023048054426908493, 0.5911130309104919, 0.0..."
13,function,parent,"[0.7450680857100948, 0.6734723524136436, 0.664...","[0.5921923539855264, 0.46599123266641984, 0.49...","[0.8519933405247602, 0.82915261558124, 0.80579..."
14,fun,parent,"[0.42034617563088733, 0.3899216913152486, 0.64...","[0.03860512375831604, 0.01738902637735009, 0.3...","[0.9826384782791138, 0.9017252445220947, 0.985..."
16,=,leaf,"[0.8114004731178284, 0.9927055239677429, 0.717...","[0.8114004731178284, 0.9927055239677429, 0.717...","[0.8114004731178284, 0.9927055239677429, 0.717..."
18,variable,leaf,"[0.00041297078545454724, 0.6361809223890305, 0...","[1.8452654160228121e-07, 0.18446189165115356, ...","[0.0008257570443674922, 0.9996611326932907, 0...."
43,exp_infix,parent,"[0.7395694156487783, 0.6860946081578732, 0.791...","[0.35459254185358685, 0.3133827621738116, 0.44...","[0.9797844489415487, 0.9975389242172241, 0.980..."
47,(,leaf,"[0.001986368792131543, 0.8219176530838013, 0.4...","[0.001986368792131543, 0.8219176530838013, 0.4...","[0.001986368792131543, 0.8219176530838013, 0.4..."
53,exp_name,parent,"[0.9920405745506287, 0.5917035341262817, 0.999...","[0.9920405745506287, 0.5917035341262817, 0.999...","[0.9920405745506287, 0.5917035341262817, 0.999..."


In [19]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [20]:
create_folder(params['output_path'] + '/' + params['current_model'] + '_q_' + params['quantization'])
global_concept_dataframe.to_csv(params['output_path'] + '/' + params['current_model'] + '_q_' + params['quantization'] + '/' + 'global_analysis.csv')

In [21]:
torch.cuda.empty_cache()
gc.collect()

20